# BettingEdge v2 — Weekly Prediction Development Notebook

This is the Google Colab development version of the weekly prediction pipeline. It was used to build and test the feature engineering logic that now lives in `predict_betting.ipynb`.

> **Note:** Paths reference Google Drive. This notebook is for reference/development only — use `predict_betting.ipynb` for production runs.

In [76]:
import joblib
from dataclasses import dataclass, field
from typing import Tuple, Dict, Any

## Load Production Model

Loads `betting_model.pkl` — the trained XGBoost sklearn Pipeline. Inspects the preprocessor to identify the exact feature columns expected by the model.

In [77]:
@dataclass(frozen=True)
class FinalCfg:
    test_size: float = 0.2
    random_state: int = 42
    oof_splits: int = 5
    weight_win: float = 2.0
    weight_loss: float = 1.0
    drop_non_features: Tuple[str, ...] = ('game_id','home_team','away_team','season','week')
    categorical_cols: Tuple[str, ...] = ('roof','surface')
    boolean_cols: Tuple[str, ...] = (
        'is_playoff','is_final_week','home_qb_switch','away_qb_switch','is_home_qb_new','is_away_qb_new'
    )
    # For OOF weighting model
    base_xgb_params: Dict[str, Any] = field(default_factory=lambda: dict(
        n_estimators=500, max_depth=3, learning_rate=0.01, min_child_weight=3,
        subsample=0.6, colsample_bytree=0.6, reg_alpha=1.0, reg_lambda=3.0,
        objective='reg:squarederror', random_state=42, tree_method='hist', n_jobs=1
    ))

In [78]:
res = joblib.load('/content/drive/MyDrive/BettingEdgeContinued/fantasy_model.pkl')

In [79]:
print(res.keys())
# → dict_keys(['mae', 'r2', 'y_test', 'y_pred', 'pipeline', 'used_columns', 'config'])

dict_keys(['mae', 'r2', 'y_test', 'y_pred', 'pipeline', 'used_columns', 'config'])


In [80]:
# The pipeline is what actually makes predictions
pipeline = res["pipeline"]
pipeline

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('cat',
                                                  OrdinalEncoder(handle_unknown='use_encoded_value',
                                                                 unknown_value=-1),
                                                  ['roof', 'surface']),
                                                 ('num', 'passthrough',
                                                  ['spread_line', 'away_rest',
                                                   'home_rest', 'total_line',
                                                   'div_game', 'temp', 'wind',
                                                   'home_rolling_avg_epa',
                                                   'home_rolling_avg_yards',
                                                   'home_rolling_play_count',
                                                   'away_rolling_avg_e...
                              feature_types=None, feature_weights=None,
                              gamma=None, grow_policy=None,
                              importance_type=None,
                              interaction_constraints=None, learning_rate=0.01,
                              max_bin=None, max_cat_threshold=None,
                              max_cat_to_onehot=None, max_delta_step=None,
                              max_depth=2, max_leaves=None, min_child_weight=3,
                              missing=nan, monotone_constraints=None,
                              multi_strategy=None, n_estimators=384, n_jobs=1,
                              num_parallel_tree=None, ...))])

In [81]:
# This tells you EXACTLY what columns the model was trained on
pre = pipeline.named_steps["preprocessor"]
pre

ColumnTransformer(transformers=[('cat',
                                 OrdinalEncoder(handle_unknown='use_encoded_value',
                                                unknown_value=-1),
                                 ['roof', 'surface']),
                                ('num', 'passthrough',
                                 ['spread_line', 'away_rest', 'home_rest',
                                  'total_line', 'div_game', 'temp', 'wind',
                                  'home_rolling_avg_epa',
                                  'home_rolling_avg_yards',
                                  'home_rolling_play_count',
                                  'away_rolling_avg_epa',
                                  'away_rolling_avg_yards',
                                  'aw...
                                  'epa_home_def_away_off_rolling_diff',
                                  'avg_yards_home_off_away_def_rolling_diff',
                                  'avg_yards_home_def_away_off_rolling_diff',
                                  'play_count_home_off_away_def_rolling_diff',
                                  'play_count_home_def_away_off_rolling_diff',
                                  'home_recent_sos_opponent_avg',
                                  'home_season_sos_opponent_avg',
                                  'away_recent_sos_opponent_avg',
                                  'away_season_sos_opponent_avg', 'sos_diff', ...])],
                  verbose_feature_names_out=False)

In [82]:
cat_cols  = pre.transformers_[0][2]   # OrdinalEncoder columns (roof, surface)
num_cols  = pre.transformers_[1][2]   # passthrough columns (everything else)

In [83]:
print("Categorical features:", cat_cols)
print("Numeric/Boolean features:", num_cols)
print("\nTotal features expected:", len(cat_cols) + len(num_cols))

Categorical features: ['roof', 'surface']
Numeric/Boolean features: ['spread_line', 'away_rest', 'home_rest', 'total_line', 'div_game', 'temp', 'wind', 'home_rolling_avg_epa', 'home_rolling_avg_yards', 'home_rolling_play_count', 'away_rolling_avg_epa', 'away_rolling_avg_yards', 'away_rolling_play_count', 'home_rolling_allowed_avg_epa', 'home_rolling_allowed_avg_yards', 'home_rolling_allowed_play_count', 'away_rolling_allowed_avg_epa', 'away_rolling_allowed_avg_yards', 'away_rolling_allowed_play_count', 'epa_home_off_away_def_rolling_diff', 'epa_home_def_away_off_rolling_diff', 'avg_yards_home_off_away_def_rolling_diff', 'avg_yards_home_def_away_off_rolling_diff', 'play_count_home_off_away_def_rolling_diff', 'play_count_home_def_away_off_rolling_diff', 'home_recent_sos_opponent_avg', 'home_season_sos_opponent_avg', 'away_recent_sos_opponent_avg', 'away_season_sos_opponent_avg', 'sos_diff', 'season_sos_diff', 'home_allpro_last_3_years_weighted', 'away_allpro_last_3_years_weighted', 'diff

In [84]:
pip install nflreadpy

In [85]:
import nflreadpy as nfl
import pandas as pd

raw = nfl.load_schedules([2025])

print(type(raw))   # confirms it's a polars DataFrame

# Convert Polars → Pandas correctly
schedule = raw.to_pandas()

print(schedule.shape)
print(schedule.columns.tolist())

<class 'polars.dataframe.frame.DataFrame'>
(285, 46)
['game_id', 'season', 'game_type', 'week', 'gameday', 'weekday', 'gametime', 'away_team', 'away_score', 'home_team', 'home_score', 'location', 'result', 'total', 'overtime', 'old_game_id', 'gsis', 'nfl_detail_id', 'pfr', 'pff', 'espn', 'ftn', 'away_rest', 'home_rest', 'away_moneyline', 'home_moneyline', 'spread_line', 'away_spread_odds', 'home_spread_odds', 'total_line', 'under_odds', 'over_odds', 'div_game', 'roof', 'surface', 'temp', 'wind', 'away_qb_id', 'home_qb_id', 'away_qb_name', 'home_qb_name', 'away_coach', 'home_coach', 'referee', 'stadium_id', 'stadium']


## Schedule & Basic Feature Wrangling

Loads the 2025 NFL schedule from `nflreadpy`, filters to the target week, and builds the base feature set (spread, roof, surface, rest days, etc.).

In [86]:
key_cols = [
    'game_id', 'home_team', 'away_team', 'week', 'season',
    'spread_line', 'total_line', 'away_rest', 'home_rest',
    'div_game', 'roof', 'surface', 'temp', 'wind', 'result'
]

for col in key_cols:
    status = "✅" if col in schedule.columns else "❌ MISSING"
    print(f"{status}  {col}")

print("\n\nAll available columns:")
print(schedule.columns.tolist())

✅  game_id
✅  home_team
✅  away_team
✅  week
✅  season
✅  spread_line
✅  total_line
✅  away_rest
✅  home_rest
✅  div_game
✅  roof
✅  surface
✅  temp
✅  wind
✅  result


All available columns:
['game_id', 'season', 'game_type', 'week', 'gameday', 'weekday', 'gametime', 'away_team', 'away_score', 'home_team', 'home_score', 'location', 'result', 'total', 'overtime', 'old_game_id', 'gsis', 'nfl_detail_id', 'pfr', 'pff', 'espn', 'ftn', 'away_rest', 'home_rest', 'away_moneyline', 'home_moneyline', 'spread_line', 'away_spread_odds', 'home_spread_odds', 'total_line', 'under_odds', 'over_odds', 'div_game', 'roof', 'surface', 'temp', 'wind', 'away_qb_id', 'home_qb_id', 'away_qb_name', 'home_qb_name', 'away_coach', 'home_coach', 'referee', 'stadium_id', 'stadium']


In [87]:
TARGET_WEEK = 10  # change this each week you want to predict

week_games = schedule[schedule['week'] == TARGET_WEEK]

# Show just the columns relevant to us
cols = ['game_id', 'home_team', 'away_team', 'gameday', 'gametime',
        'spread_line', 'total_line', 'away_rest', 'home_rest',
        'div_game', 'roof', 'surface', 'temp', 'wind']

print(week_games[cols].to_string())

             game_id home_team away_team     gameday gametime  spread_line  total_line  away_rest  home_rest  div_game      roof     surface  temp  wind
135   2025_10_LV_DEN       DEN        LV  2025-11-06    20:15          9.5        42.5          4          4         1  outdoors       grass  60.0  10.0
136  2025_10_ATL_IND       IND       ATL  2025-11-09    09:30          6.5        48.5          7          7         0    closed       grass  46.0   2.0
137   2025_10_NO_CAR       CAR        NO  2025-11-09    13:00          5.5        38.5          7          7         1  outdoors       grass  73.0  15.0
138  2025_10_NYG_CHI       CHI       NYG  2025-11-09    13:00          4.5        45.5          7          7         0  outdoors       grass  33.0  10.0
139  2025_10_JAX_HOU       HOU       JAX  2025-11-09    13:00         -1.5        37.5          7          7         1    closed   astroturf   NaN   NaN
140  2025_10_BUF_MIA       MIA       BUF  2025-11-09    13:00         -8.5        

In [88]:
# All completed games BEFORE the target week — used to build rolling features
history = schedule[
    (schedule['week'] < TARGET_WEEK) &
    (schedule['result'].notna())   # result is NaN for games not yet played
].copy()

# The games you actually want to predict
upcoming = schedule[schedule['week'] == TARGET_WEEK].copy()

# Dome games have no temp/wind — fill with season medians from history
upcoming['temp'] = upcoming['temp'].fillna(72)
upcoming['wind'] = upcoming['wind'].fillna(0)

print(f"History: {len(history)} completed games (weeks 1 through {TARGET_WEEK - 1})")
print(f"Upcoming: {len(upcoming)} games to predict in week {TARGET_WEEK}")

History: 135 completed games (weeks 1 through 9)
Upcoming: 14 games to predict in week 10


In [89]:
# These are the Group 1 features the model expects
group1_features = [
    'spread_line', 'away_rest', 'home_rest', 'total_line',
    'div_game', 'temp', 'wind', 'roof', 'surface'
]

print("Group 1 feature check for upcoming games:\n")
print(upcoming[group1_features].to_string())
print("\nNull counts:")
print(upcoming[group1_features].isnull().sum())

Group 1 feature check for upcoming games:

     spread_line  away_rest  home_rest  total_line  div_game  temp  wind      roof     surface
135          9.5          4          4        42.5         1  60.0  10.0  outdoors       grass
136          6.5          7          7        48.5         0  46.0   2.0    closed       grass
137          5.5          7          7        38.5         1  73.0  15.0  outdoors       grass
138          4.5          7          7        45.5         0  33.0  10.0  outdoors       grass
139         -1.5          7          7        37.5         1  72.0   0.0    closed   astroturf
140         -8.5          7         10        50.5         1  84.0   6.0  outdoors       grass
141         -4.5         10          7        48.5         0  72.0   0.0      dome   sportturf
142         -1.5         14         14        37.5         0  62.0  12.0  outdoors   fieldturf
143          2.5          7         14        48.5         0  82.0  11.0  outdoors       grass
144    

## Play-By-Play Rolling Features (EPA, Yards)

Loads PBP data and computes per-team rolling offensive and defensive EPA and yards/play. Uses a shift-then-roll pattern to avoid leaking current-game stats.

In [90]:
raw_pbp = nfl.load_pbp([2025])
pbp = raw_pbp.to_pandas()

print(pbp.shape)
print(pbp['play_type'].value_counts())

(48771, 372)
play_type
pass           19735
run            14893
no_play         4732
kickoff         2918
punt            2042
extra_point     1324
field_goal      1140
qb_kneel         452
qb_spike          80
Name: count, dtype: int64


In [91]:
pbp_rp = pbp[
    pbp['play_type'].isin(['run', 'pass']) &
    pbp['posteam'].notna() &
    pbp['defteam'].notna()
].copy()

print(pbp_rp.shape)
print(pbp_rp[['game_id', 'posteam', 'defteam', 'epa', 'yards_gained', 'play_type']].head(10))

(34628, 372)
           game_id posteam defteam       epa  yards_gained play_type
2   2025_01_ARI_NO     ARI      NO -0.190052           3.0       run
3   2025_01_ARI_NO     ARI      NO  1.317340          11.0      pass
4   2025_01_ARI_NO     ARI      NO -1.694360         -11.0      pass
5   2025_01_ARI_NO     ARI      NO -1.284150          -2.0       run
6   2025_01_ARI_NO     ARI      NO -0.840574           1.0       run
8   2025_01_ARI_NO      NO     ARI -0.194728           3.0       run
9   2025_01_ARI_NO      NO     ARI -0.788527           0.0      pass
10  2025_01_ARI_NO      NO     ARI -1.545796           0.0      pass
12  2025_01_ARI_NO     ARI      NO  0.041066           5.0      pass
13  2025_01_ARI_NO     ARI      NO  1.133550          13.0       run


In [92]:
# Offense stats — grouped by game and the team with the ball
off_stats = (
    pbp_rp
    .groupby(['game_id', 'posteam'])
    .agg(
        avg_epa=('epa', 'mean'),
        avg_yards=('yards_gained', 'mean'),
        play_count=('play_id', 'count')
    )
    .reset_index()
    .rename(columns={'posteam': 'team'})
)

# Defense stats — grouped by game and the team defending
def_stats = (
    pbp_rp
    .groupby(['game_id', 'defteam'])
    .agg(
        allowed_avg_epa=('epa', 'mean'),
        allowed_avg_yards=('yards_gained', 'mean'),
        allowed_play_count=('play_id', 'count')
    )
    .reset_index()
    .rename(columns={'defteam': 'team'})
)

print("Offense stats shape:", off_stats.shape)
print(off_stats.head())

Offense stats shape: (570, 5)
           game_id team   avg_epa  avg_yards  play_count
0   2025_01_ARI_NO  ARI  0.043989   4.524590          61
1   2025_01_ARI_NO   NO -0.041654   4.701493          67
2  2025_01_BAL_BUF  BAL  0.385699   8.640000          50
3  2025_01_BAL_BUF  BUF  0.234987   6.519481          77
4  2025_01_CAR_JAX  CAR -0.256380   4.266667          60


In [93]:
# Pull just the week/season info from the schedule to attach to pbp stats
week_lookup = schedule[['game_id', 'week', 'season']].drop_duplicates()

off_stats = off_stats.merge(week_lookup, on='game_id', how='left')
def_stats = def_stats.merge(week_lookup, on='game_id', how='left')

print(off_stats[['game_id', 'team', 'week', 'avg_epa']].head(10))

           game_id team  week   avg_epa
0   2025_01_ARI_NO  ARI     1  0.043989
1   2025_01_ARI_NO   NO     1 -0.041654
2  2025_01_BAL_BUF  BAL     1  0.385699
3  2025_01_BAL_BUF  BUF     1  0.234987
4  2025_01_CAR_JAX  CAR     1 -0.256380
5  2025_01_CAR_JAX  JAX     1  0.152458
6  2025_01_CIN_CLE  CIN     1 -0.108868
7  2025_01_CIN_CLE  CLE     1 -0.051202
8  2025_01_DAL_PHI  DAL     1  0.035278
9  2025_01_DAL_PHI  PHI     1  0.160450


In [94]:
# Sort by team and week so the rolling window goes in the right direction
off_stats = off_stats.sort_values(['team', 'season', 'week']).reset_index(drop=True)
def_stats = def_stats.sort_values(['team', 'season', 'week']).reset_index(drop=True)

# Rolling offense — shift(1) means "not including the current game"
# This is critical — you can't use a game's own stats to predict that game
for feat in ['avg_epa', 'avg_yards', 'play_count']:
    off_stats[f'rolling_{feat}'] = (
        off_stats
        .groupby('team')[feat]
        .transform(lambda x: x.shift(1).rolling(5, min_periods=1).mean())
    )

# Rolling defense
for feat in ['allowed_avg_epa', 'allowed_avg_yards', 'allowed_play_count']:
    def_stats[f'rolling_{feat}'] = (
        def_stats
        .groupby('team')[feat]
        .transform(lambda x: x.shift(1).rolling(5, min_periods=1).mean())
    )

print("Offense rolling stats:")
print(off_stats[['team', 'week', 'avg_epa', 'rolling_avg_epa']].head(15))

Offense rolling stats:
   team  week   avg_epa  rolling_avg_epa
0   ARI     1  0.043989              NaN
1   ARI     2  0.127011         0.043989
2   ARI     3 -0.011040         0.085500
3   ARI     4 -0.081304         0.053320
4   ARI     5 -0.074265         0.019664
5   ARI     6  0.089873         0.000878
6   ARI     7 -0.052211         0.010055
7   ARI     9  0.113321        -0.025789
8   ARI    10 -0.210903        -0.000917
9   ARI    11  0.061463        -0.026837
10  ARI    12 -0.146952         0.000309
11  ARI    13 -0.026304        -0.047056
12  ARI    14 -0.061863        -0.041875
13  ARI    15  0.012079        -0.076912
14  ARI    16 -0.004240        -0.032315


In [95]:
# We only need the rolling columns, not the raw game-level stats
off_rolling = off_stats[['game_id', 'team', 'rolling_avg_epa', 'rolling_avg_yards', 'rolling_play_count']]
def_rolling = def_stats[['game_id', 'team', 'rolling_allowed_avg_epa', 'rolling_allowed_avg_yards', 'rolling_allowed_play_count']]

# Merge home team offense
upcoming = upcoming.merge(
    off_rolling.rename(columns={'team': 'home_team', **{c: f'home_{c}' for c in off_rolling.columns if c.startswith('rolling')}}),
    on=['game_id', 'home_team'], how='left'
)

# Merge away team offense
upcoming = upcoming.merge(
    off_rolling.rename(columns={'team': 'away_team', **{c: f'away_{c}' for c in off_rolling.columns if c.startswith('rolling')}}),
    on=['game_id', 'away_team'], how='left'
)

# Merge home team defense
upcoming = upcoming.merge(
    def_rolling.rename(columns={'team': 'home_team', **{c: f'home_{c}' for c in def_rolling.columns if c.startswith('rolling')}}),
    on=['game_id', 'home_team'], how='left'
)

# Merge away team defense
upcoming = upcoming.merge(
    def_rolling.rename(columns={'team': 'away_team', **{c: f'away_{c}' for c in def_rolling.columns if c.startswith('rolling')}}),
    on=['game_id', 'away_team'], how='left'
)

print(upcoming[['home_team', 'away_team', 'home_rolling_avg_epa', 'away_rolling_avg_epa',
                 'home_rolling_allowed_avg_epa', 'away_rolling_allowed_avg_epa']].to_string())

   home_team away_team  home_rolling_avg_epa  away_rolling_avg_epa  home_rolling_allowed_avg_epa  away_rolling_allowed_avg_epa
0        DEN        LV              0.069126             -0.223643                     -0.077516                      0.046081
1        IND       ATL              0.223933             -0.010268                     -0.047286                      0.040593
2        CAR        NO              0.001531             -0.216599                      0.081096                      0.015716
3        CHI       NYG              0.123259              0.068929                     -0.022078                      0.164295
4        HOU       JAX              0.063374              0.016247                     -0.210792                      0.086189
5        MIA       BUF             -0.085888              0.138325                      0.011105                     -0.054874
6        MIN       BAL             -0.090772             -0.060594                      0.133357               

In [96]:
# These were computed in your notebook as combinations of offense vs opponent defense
upcoming['epa_home_off_away_def_rolling_diff'] = (
    upcoming['home_rolling_avg_epa'] - upcoming['away_rolling_allowed_avg_epa']
)
upcoming['epa_home_def_away_off_rolling_diff'] = (
    upcoming['home_rolling_allowed_avg_epa'] - upcoming['away_rolling_avg_epa']
)
upcoming['avg_yards_home_off_away_def_rolling_diff'] = (
    upcoming['home_rolling_avg_yards'] - upcoming['away_rolling_allowed_avg_yards']
)
upcoming['avg_yards_home_def_away_off_rolling_diff'] = (
    upcoming['home_rolling_allowed_avg_yards'] - upcoming['away_rolling_avg_yards']
)
upcoming['play_count_home_off_away_def_rolling_diff'] = (
    upcoming['home_rolling_play_count'] - upcoming['away_rolling_allowed_play_count']
)
upcoming['play_count_home_def_away_off_rolling_diff'] = (
    upcoming['home_rolling_allowed_play_count'] - upcoming['away_rolling_play_count']
)

# Sanity check
diff_cols = [c for c in upcoming.columns if 'diff' in c]
print(upcoming[['home_team', 'away_team'] + diff_cols].to_string())

   home_team away_team  epa_home_off_away_def_rolling_diff  epa_home_def_away_off_rolling_diff  avg_yards_home_off_away_def_rolling_diff  avg_yards_home_def_away_off_rolling_diff  play_count_home_off_away_def_rolling_diff  play_count_home_def_away_off_rolling_diff
0        DEN        LV                            0.023046                            0.146127                                  0.574017                                 -0.620338                                       -2.4                                       12.0
1        IND       ATL                            0.183339                           -0.037018                                  1.095865                                 -0.690890                                       -0.4                                        9.4
2        CAR        NO                           -0.014186                            0.297695                                  0.049364                                  0.192464                           

In [97]:
# How many of the 79 features do I have so far?
model_features = cat_cols + num_cols   # from step 1 earlier

have = [f for f in model_features if f in upcoming.columns]
missing = [f for f in model_features if f not in upcoming.columns]

print(f"Have: {len(have)}/79")
print(f"\nStill missing ({len(missing)}):")
for f in missing:
    print(f"  - {f}")

Have: 27/79

Still missing (52):
  - home_recent_sos_opponent_avg
  - home_season_sos_opponent_avg
  - away_recent_sos_opponent_avg
  - away_season_sos_opponent_avg
  - sos_diff
  - season_sos_diff
  - home_allpro_last_3_years_weighted
  - away_allpro_last_3_years_weighted
  - diff_allpro_last_3_years_weighted
  - home_allpro_prev_year
  - away_allpro_prev_year
  - diff_allpro_prev_year
  - home_offense_allpro_3_years
  - away_offense_allpro_3_years
  - home_defense_allpro_3_years
  - away_defense_allpro_3_years
  - allpro_diff_home_off_away_def_3_years
  - allpro_diff_home_def_away_off_3_years 
  - home_offense_allpro_prev_year
  - away_offense_allpro_prev_year
  - home_defense_allpro_prev_year
  - away_defense_allpro_prev_year
  - allpro_diff_home_off_away_def_prev_year
  - allpro_diff_home_def_away_off_prev_year
  - league_rolling_avg_abs_margin_by_week
  - home_qbr_prev_year
  - away_qbr_prev_year
  - diff_qbr_prev_year
  - home_injured_count
  - away_injured_count
  - diff_injured

## Historical Win % & Strength of Schedule (SOS)

Converts the schedule into long format (one row per team per game), computes cumulative and rolling win %, and derives SOS as the average win % of the last 3 opponents.

In [98]:
# Split each game into two rows — one for home team, one for away team
home_games = history[['season', 'week', 'home_team', 'away_team', 'home_score', 'away_score']].copy()
home_games.columns = ['season', 'week', 'team', 'opponent', 'team_score', 'opp_score']

away_games = history[['season', 'week', 'away_team', 'home_team', 'away_score', 'home_score']].copy()
away_games.columns = ['season', 'week', 'team', 'opponent', 'team_score', 'opp_score']

long_df = pd.concat([home_games, away_games], ignore_index=True)
long_df = long_df.sort_values(['team', 'season', 'week']).reset_index(drop=True)

# Mark wins
long_df['team_win'] = (long_df['team_score'] > long_df['opp_score']).astype(int)

print(long_df.head(10))
print(long_df.shape)

   season  week team opponent  team_score  opp_score  team_win
0    2025     1  ARI       NO          20         13         1
1    2025     2  ARI      CAR          27         22         1
2    2025     3  ARI       SF          15         16         0
3    2025     4  ARI      SEA          20         23         0
4    2025     5  ARI      TEN          21         22         0
5    2025     6  ARI      IND          27         31         0
6    2025     7  ARI       GB          23         27         0
7    2025     9  ARI      DAL          27         17         1
8    2025     1  ATL       TB          20         23         0
9    2025     2  ATL      MIN          22          6         1
(270, 7)


In [99]:
# shift() again — same reason as rolling PBP stats
# A team's win % entering a game can't include that game itself
long_df['win_pct'] = (
    long_df
    .groupby('team')['team_win']
    .transform(lambda x: x.shift(1).expanding().mean())
)

print(long_df[['team', 'week', 'team_win', 'win_pct']].head(20))

   team  week  team_win   win_pct
0   ARI     1         1       NaN
1   ARI     2         1  1.000000
2   ARI     3         0  1.000000
3   ARI     4         0  0.666667
4   ARI     5         0  0.500000
5   ARI     6         0  0.400000
6   ARI     7         0  0.333333
7   ARI     9         1  0.285714
8   ATL     1         0       NaN
9   ATL     2         1  0.000000
10  ATL     3         0  0.500000
11  ATL     4         1  0.333333
12  ATL     6         1  0.500000
13  ATL     7         0  0.600000
14  ATL     8         0  0.500000
15  ATL     9         0  0.428571
16  BAL     1         0       NaN
17  BAL     2         1  0.000000
18  BAL     3         0  0.500000
19  BAL     4         0  0.333333


In [100]:
# Build a lookup: for each team/week, what is their win %?
win_pct_lookup = long_df[['season', 'week', 'team', 'win_pct']].copy()
win_pct_lookup.columns = ['season', 'week', 'opponent', 'opponent_win_pct']

# Merge opponent's win % onto each row
long_df = long_df.merge(win_pct_lookup, on=['season', 'week', 'opponent'], how='left')

print(long_df[['team', 'week', 'opponent', 'win_pct', 'opponent_win_pct']].head(20))

   team  week opponent   win_pct  opponent_win_pct
0   ARI     1       NO       NaN               NaN
1   ARI     2      CAR  1.000000          0.000000
2   ARI     3       SF  1.000000          1.000000
3   ARI     4      SEA  0.666667          0.666667
4   ARI     5      TEN  0.500000          0.000000
5   ARI     6      IND  0.400000          0.800000
6   ARI     7       GB  0.333333          0.600000
7   ARI     9      DAL  0.285714          0.375000
8   ATL     1       TB       NaN               NaN
9   ATL     2      MIN  0.000000          1.000000
10  ATL     3      CAR  0.500000          0.000000
11  ATL     4      WAS  0.333333          0.666667
12  ATL     6      BUF  0.500000          0.800000
13  ATL     7       SF  0.600000          0.666667
14  ATL     8      MIA  0.500000          0.142857
15  ATL     9       NE  0.428571          0.750000
16  BAL     1      BUF       NaN               NaN
17  BAL     2      CLE  0.000000          0.000000
18  BAL     3      DET  0.50000

In [101]:
# Recent SOS = avg win % of last 3 opponents
long_df['recent_sos'] = (
    long_df
    .groupby('team')['opponent_win_pct']
    .transform(lambda x: x.shift(1).rolling(3, min_periods=1).mean().fillna(0))
)

# Season SOS = avg win % of all opponents so far this season
long_df['season_sos'] = (
    long_df
    .groupby('team')['opponent_win_pct']
    .transform(lambda x: x.shift(1).expanding().mean().fillna(0))
)

print(long_df[['team', 'week', 'recent_sos', 'season_sos']].head(20))

   team  week  recent_sos  season_sos
0   ARI     1    0.000000    0.000000
1   ARI     2    0.000000    0.000000
2   ARI     3    0.000000    0.000000
3   ARI     4    0.500000    0.500000
4   ARI     5    0.555556    0.555556
5   ARI     6    0.555556    0.416667
6   ARI     7    0.488889    0.493333
7   ARI     9    0.466667    0.511111
8   ATL     1    0.000000    0.000000
9   ATL     2    0.000000    0.000000
10  ATL     3    1.000000    1.000000
11  ATL     4    0.500000    0.500000
12  ATL     6    0.555556    0.555556
13  ATL     7    0.488889    0.616667
14  ATL     8    0.711111    0.626667
15  ATL     9    0.536508    0.546032
16  BAL     1    0.000000    0.000000
17  BAL     2    0.000000    0.000000
18  BAL     3    0.000000    0.000000
19  BAL     4    0.250000    0.250000


In [102]:
sos_lookup = long_df[['season', 'week', 'team', 'recent_sos', 'season_sos']].copy()

# We need the SOS for the TARGET week — meaning what was each team's SOS entering that week
# Since long_df only has history, we need the last known value per team
latest_sos = (
    sos_lookup
    .sort_values(['team', 'season', 'week'])
    .groupby('team')
    .last()
    .reset_index()
    [['team', 'recent_sos', 'season_sos']]
)

# Merge for home team
upcoming = upcoming.merge(
    latest_sos.rename(columns={
        'team': 'home_team',
        'recent_sos': 'home_recent_sos_opponent_avg',
        'season_sos': 'home_season_sos_opponent_avg'
    }),
    on='home_team', how='left'
)

# Merge for away team
upcoming = upcoming.merge(
    latest_sos.rename(columns={
        'team': 'away_team',
        'recent_sos': 'away_recent_sos_opponent_avg',
        'season_sos': 'away_season_sos_opponent_avg'
    }),
    on='away_team', how='left'
)

# Fill any NaN with 0 (same as notebook)
for col in ['home_recent_sos_opponent_avg', 'home_season_sos_opponent_avg',
            'away_recent_sos_opponent_avg', 'away_season_sos_opponent_avg']:
    upcoming[col] = upcoming[col].fillna(0)

print(upcoming[['home_team', 'away_team',
                'home_recent_sos_opponent_avg', 'away_recent_sos_opponent_avg',
                'home_season_sos_opponent_avg', 'away_season_sos_opponent_avg']].to_string())

   home_team away_team  home_recent_sos_opponent_avg  away_recent_sos_opponent_avg  home_season_sos_opponent_avg  away_season_sos_opponent_avg
0        DEN        LV                      0.253968                      0.483333                      0.632653                      0.547222
1        IND       ATL                      0.403175                      0.536508                      0.446599                      0.546032
2        CAR        NO                      0.355556                      0.638095                      0.450000                      0.666327
3        CHI       NYG                      0.311111                      0.726984                      0.294444                      0.454422
4        HOU       JAX                      0.543651                      0.588889                      0.521825                      0.627778
5        MIA       BUF                      0.422222                      0.523810                      0.359524                      0.261905

In [103]:
upcoming['sos_diff'] = (
    upcoming['home_recent_sos_opponent_avg'] - upcoming['away_recent_sos_opponent_avg']
)
upcoming['season_sos_diff'] = (
    upcoming['home_season_sos_opponent_avg'] - upcoming['away_season_sos_opponent_avg']
)

print(upcoming[['home_team', 'away_team', 'sos_diff', 'season_sos_diff']].to_string())

   home_team away_team  sos_diff  season_sos_diff
0        DEN        LV -0.229365         0.085431
1        IND       ATL -0.133333        -0.099433
2        CAR        NO -0.282540        -0.216327
3        CHI       NYG -0.415873        -0.159977
4        HOU       JAX -0.045238        -0.105952
5        MIA       BUF -0.101587         0.097619
6        MIN       BAL -0.009524         0.134127
7        NYJ       CLE -0.022222         0.044444
8         TB        NE  0.521429         0.181009
9        SEA       ARI  0.183333         0.091667
10        SF        LA  0.038889        -0.009921
11       WAS       DET -0.109524         0.141950
12       LAC       PIT  0.111111         0.110317
13        GB       PHI  0.033333        -0.094444


In [104]:
have = [f for f in model_features if f in upcoming.columns]
missing = [f for f in model_features if f not in upcoming.columns]

print(f"Have: {len(have)}/79")
print(f"\nStill missing ({len(missing)}):")
for f in missing:
    print(f"  - {f}")

Have: 33/79

Still missing (46):
  - home_allpro_last_3_years_weighted
  - away_allpro_last_3_years_weighted
  - diff_allpro_last_3_years_weighted
  - home_allpro_prev_year
  - away_allpro_prev_year
  - diff_allpro_prev_year
  - home_offense_allpro_3_years
  - away_offense_allpro_3_years
  - home_defense_allpro_3_years
  - away_defense_allpro_3_years
  - allpro_diff_home_off_away_def_3_years
  - allpro_diff_home_def_away_off_3_years 
  - home_offense_allpro_prev_year
  - away_offense_allpro_prev_year
  - home_defense_allpro_prev_year
  - away_defense_allpro_prev_year
  - allpro_diff_home_off_away_def_prev_year
  - allpro_diff_home_def_away_off_prev_year
  - league_rolling_avg_abs_margin_by_week
  - home_qbr_prev_year
  - away_qbr_prev_year
  - diff_qbr_prev_year
  - home_injured_count
  - away_injured_count
  - diff_injured_count
  - diff_active_allpro_weighted
  - diff_active_allpro_prev_year
  - home_rolling_win_pct
  - away_rolling_win_pct
  - sack_diff
  - sack_diff_reverse
  - tur

## All-Pro Roster Quality Features

Loads `nfl_allpro_1997_2025.csv` and builds a weighted 3-year lookback of All-Pro selections (weight 4/2/1). Separate offense and defense counts; computes home-minus-away difference features.

In [105]:
allpro_df = pd.read_csv('/content/drive/MyDrive/BettingEdgeContinued/nfl_allpro_1997_2025.csv')  # update path to wherever your CSV lives
allpro_df = allpro_df[allpro_df['Team'] != '2TM'].copy()
print(allpro_df.shape)


# Confirm they're gone
print(f"\nTotal rows after dropping 2TM: {len(allpro_df)}")

print(allpro_df.columns.tolist())
print(allpro_df.head(10))
print(allpro_df['Year'].value_counts().sort_index())

(2043, 5)

Total rows after dropping 2TM: 2043
['Pos', 'Player', 'Team', 'Year', 'Side']
  Pos          Player Team  Year     Side
0  QB     Steve Young  SFO  1997  offense
1  FB    Mike Alstott  TAM  1997  offense
2  FB     Charles Way  NYG  1997  offense
3   T     James Hasty  KAN  1997  offense
4   T     Chuck Smith  ATL  1997  offense
5   T     Dale Carter  KAN  1997  offense
6   T  Robert Porcher  DET  1997  offense
7   G    Will Shields  KAN  1997  offense
8  DE      Neil Smith  DEN  1997  defense
9  DE    Reggie White  GNB  1997  defense
Year
1997    64
1998    67
1999    62
2000    63
2001    66
2002    64
2003    60
2004    66
2005    61
2006    54
2007    64
2008    61
2009    84
2010    88
2011    87
2012    76
2013    84
2014    77
2015    78
2016    77
2017    77
2018    65
2019    64
2020    74
2021    70
2022    68
2023    76
2024    73
2025    73
Name: count, dtype: int64


In [106]:
team_map = {
    'STL': 'LA',    # Rams in St. Louis → now LA (not LAR, schedule uses LA)
    'LAR': 'LA',    # some years PFR uses LAR → schedule uses LA
    'OAK': 'LV',    # Raiders Oakland
    'LVR': 'LV',    # Raiders Las Vegas — PFR uses LVR, schedule uses LV
    'SD':  'LAC',   # Chargers San Diego
    'SDG': 'LAC',   # PFR alternate abbreviation for San Diego
    'NWE': 'NE',
    'KAN': 'KC',
    'GNB': 'GB',
    'NOR': 'NO',
    'TAM': 'TB',
    'SFO': 'SF',
}

allpro_df['Team'] = allpro_df['Team'].replace(team_map)

# Re-run the check — both sets should now be empty
allpro_teams   = set(allpro_df['Team'].unique())
schedule_teams = set(schedule['home_team'].unique())

print("Still mismatched:", allpro_teams - schedule_teams)

Still mismatched: set()


In [107]:
# For each season, look back 3 years and weight by recency
# Year - 1 = weight 4, Year - 2 = weight 2, Year - 3 = weight 1
# This rewards teams with recent all-pros more than older ones

all_games = []

for year in range(2006, 2026):   # 2026 = predict for 2025 season
    curr = []
    for yrs_back, weight in zip([1, 2, 3], [4, 2, 1]):
        prev_year = year - yrs_back
        df_prev = allpro_df[allpro_df['Year'] == prev_year].copy()
        df_prev['Weight'] = weight
        df_prev['Year_target'] = year
        curr.append(df_prev)

    combined = pd.concat(curr)

    # If a player was all-pro in multiple of the last 3 years,
    # only count them once (at their highest weight)
    deduped = (
        combined
        .sort_values('Weight', ascending=False)
        .drop_duplicates(['Player', 'Year_target'])
    )

    weighted_counts = (
        deduped
        .groupby(['Year_target', 'Team'])['Weight']
        .sum()
        .reset_index()
        .rename(columns={'Weight': 'allpro_weighted', 'Year_target': 'season'})
    )

    all_games.append(weighted_counts)

weighted_allpro_df = pd.concat(all_games, ignore_index=True)

print(weighted_allpro_df.head(10))
print(weighted_allpro_df[weighted_allpro_df['season'] == 2025].sort_values('allpro_weighted', ascending=False).head(10))

   season Team  allpro_weighted
0    2006  ARI                2
1    2006  ATL                9
2    2006  BAL               15
3    2006  BUF                4
4    2006  CAR               15
5    2006  CHI               25
6    2006  CIN               12
7    2006  DAL                5
8    2006  DEN                9
9    2006  DET                5
     season Team  allpro_weighted
620    2025  PHI               41
605    2025  DET               36
597    2025  BAL               28
623    2025   SF               25
604    2025  DEN               24
615    2025  MIN               21
603    2025  DAL               20
610    2025   KC               19
626    2025  WAS               16
598    2025  BUF               16


In [108]:
offense_df = allpro_df[allpro_df['Side'] == 'offense'].copy()
defense_df = allpro_df[allpro_df['Side'] == 'defense'].copy()

def build_weighted_allpro(df_ap):
    all_games = []
    for year in range(2006, 2026):
        curr = []
        for yrs_back, weight in zip([1, 2, 3], [4, 2, 1]):
            prev_year = year - yrs_back
            df_prev = df_ap[df_ap['Year'] == prev_year].copy()
            df_prev['Weight'] = weight
            df_prev['Year_target'] = year
            curr.append(df_prev)

        combined = pd.concat(curr)
        deduped = (
            combined
            .sort_values('Weight', ascending=False)
            .drop_duplicates(['Player', 'Year_target'])
        )
        weighted_counts = (
            deduped
            .groupby(['Year_target', 'Team'])['Weight']
            .sum()
            .reset_index()
            .rename(columns={'Weight': 'allpro_weighted', 'Year_target': 'season'})
        )
        all_games.append(weighted_counts)
    return pd.concat(all_games, ignore_index=True)

offense_weighted = build_weighted_allpro(offense_df)
defense_weighted = build_weighted_allpro(defense_df)

print("Offense weighted sample:")
print(offense_weighted[offense_weighted['season'] == 2025].sort_values('allpro_weighted', ascending=False).head(5))
print("\nDefense weighted sample:")
print(defense_weighted[defense_weighted['season'] == 2025].sort_values('allpro_weighted', ascending=False).head(5))

Offense weighted sample:
     season Team  allpro_weighted
537    2025  PHI               23
524    2025  DET               22
522    2025  DAL               14
538    2025   SF               14
517    2025  BAL               12

Defense weighted sample:
     season Team  allpro_weighted
553    2025  DEN               18
569    2025  PHI               18
546    2025  BAL               16
564    2025  MIN               15
554    2025  DET               14


In [109]:
prev_year_counts = (
    allpro_df
    .assign(season=allpro_df['Year'] + 1)   # shift forward — 2024 all-pros apply to 2025 season
    .groupby(['season', 'Team', 'Side'])['Player']
    .nunique()
    .reset_index(name='allpro_prev_year')
)

prev_offense = prev_year_counts[prev_year_counts['Side'] == 'offense'].drop(columns='Side')
prev_defense = prev_year_counts[prev_year_counts['Side'] == 'defense'].drop(columns='Side')

print("Prev year offense sample (2025 season):")
print(prev_offense[prev_offense['season'] == 2025].sort_values('allpro_prev_year', ascending=False).head(5))

Prev year offense sample (2025 season):
      season Team  allpro_prev_year
1063    2025  PHI                 5
1044    2025  DET                 5
1032    2025  BAL                 3
1030    2025  ATL                 2
1037    2025  CIN                 2


In [110]:
TARGET_SEASON = 2025   # match whatever season you're predicting

def merge_allpro(df, feat_df, feat_col, home_col, away_col):
    # Filter to target season and drop the season column — we don't need it after filtering
    lookup = (
        feat_df[feat_df['season'] == TARGET_SEASON]
        .drop(columns='season')   # ← this is the fix
    )

    # Home team
    df = df.merge(
        lookup.rename(columns={'Team': 'home_team', feat_col: home_col}),
        on='home_team', how='left'
    )
    # Away team
    df = df.merge(
        lookup.rename(columns={'Team': 'away_team', feat_col: away_col}),
        on='away_team', how='left'
    )
    df[home_col] = df[home_col].fillna(0)
    df[away_col] = df[away_col].fillna(0)
    return df

# Overall weighted
upcoming = merge_allpro(upcoming, weighted_allpro_df, 'allpro_weighted',
                        'home_allpro_last_3_years_weighted', 'away_allpro_last_3_years_weighted')

# Offense weighted
upcoming = merge_allpro(upcoming, offense_weighted, 'allpro_weighted',
                        'home_offense_allpro_3_years', 'away_offense_allpro_3_years')

# Defense weighted
upcoming = merge_allpro(upcoming, defense_weighted, 'allpro_weighted',
                        'home_defense_allpro_3_years', 'away_defense_allpro_3_years')

# Prev year offense
upcoming = merge_allpro(upcoming, prev_offense, 'allpro_prev_year',
                        'home_offense_allpro_prev_year', 'away_offense_allpro_prev_year')

# Prev year defense
upcoming = merge_allpro(upcoming, prev_defense, 'allpro_prev_year',
                        'home_defense_allpro_prev_year', 'away_defense_allpro_prev_year')

# Overall prev year
prev_overall = (
    allpro_df
    .assign(season=allpro_df['Year'] + 1)
    .groupby(['season', 'Team'])['Player']
    .nunique()
    .reset_index(name='allpro_prev_year')
)
upcoming = merge_allpro(upcoming, prev_overall, 'allpro_prev_year',
                        'home_allpro_prev_year', 'away_allpro_prev_year')

print(upcoming[['home_team', 'away_team',
                'home_allpro_last_3_years_weighted', 'away_allpro_last_3_years_weighted',
                'home_offense_allpro_3_years', 'away_offense_allpro_3_years',
                'home_defense_allpro_3_years', 'away_defense_allpro_3_years']].to_string())

   home_team away_team  home_allpro_last_3_years_weighted  away_allpro_last_3_years_weighted  home_offense_allpro_3_years  away_offense_allpro_3_years  home_defense_allpro_3_years  away_defense_allpro_3_years
0        DEN        LV                                 24                                  8                          6.0                          6.0                           18                            2
1        IND       ATL                                  8                                 10                          4.0                          8.0                            4                            2
2        CAR        NO                                  3                                  3                          0.0                          0.0                            3                            3
3        CHI       NYG                                  6                                  5                          2.0                          1.0              

In [117]:
upcoming['diff_allpro_last_3_years_weighted']   = upcoming['home_allpro_last_3_years_weighted']  - upcoming['away_allpro_last_3_years_weighted']
upcoming['diff_allpro_prev_year']               = upcoming['home_allpro_prev_year']               - upcoming['away_allpro_prev_year']
upcoming['allpro_diff_home_off_away_def_3_years']  = upcoming['home_offense_allpro_3_years']      - upcoming['away_defense_allpro_3_years']
upcoming['allpro_diff_home_def_away_off_3_years ']  = upcoming['home_defense_allpro_3_years']      - upcoming['away_offense_allpro_3_years']
upcoming['allpro_diff_home_off_away_def_prev_year'] = upcoming['home_offense_allpro_prev_year']   - upcoming['away_defense_allpro_prev_year']
upcoming['allpro_diff_home_def_away_off_prev_year'] = upcoming['home_defense_allpro_prev_year']   - upcoming['away_offense_allpro_prev_year']

print("Diff features:")
diff_cols = [c for c in upcoming.columns if 'diff' in c and 'allpro' in c]
print(upcoming[['home_team', 'away_team'] + diff_cols].to_string())

Diff features:
   home_team away_team  diff_allpro_last_3_years_weighted  diff_allpro_prev_year  allpro_diff_home_off_away_def_3_years  allpro_diff_home_def_away_off_3_years  allpro_diff_home_off_away_def_prev_year  allpro_diff_home_def_away_off_prev_year  allpro_diff_home_def_away_off_3_years 
0        DEN        LV                                 16                    4.0                                    4.0                                   12.0                                      1.0                                      3.0                                    12.0
1        IND       ATL                                 -2                    0.0                                    2.0                                   -4.0                                      1.0                                     -1.0                                    -4.0
2        CAR        NO                                  0                    0.0                                   -3.0                       

In [118]:
have    = [f for f in model_features if f in upcoming.columns]
missing = [f for f in model_features if f not in upcoming.columns]

print(f"Have: {len(have)}/79")
print(f"\nStill missing ({len(missing)}):")
for f in missing:
    print(f"  - {f}")

Have: 51/79

Still missing (28):
  - league_rolling_avg_abs_margin_by_week
  - home_qbr_prev_year
  - away_qbr_prev_year
  - diff_qbr_prev_year
  - home_injured_count
  - away_injured_count
  - diff_injured_count
  - diff_active_allpro_weighted
  - diff_active_allpro_prev_year
  - home_rolling_win_pct
  - away_rolling_win_pct
  - sack_diff
  - sack_diff_reverse
  - turnover_diff
  - turnover_diff_reverse
  - third_down_diff
  - third_down_diff_reverse
  - cover_rate_diff
  - scoring_diff
  - scoring_diff_reverse
  - home_coach_win_pct_prior
  - away_coach_win_pct_prior
  - is_playoff
  - is_final_week
  - home_qb_switch
  - away_qb_switch
  - is_home_qb_new
  - is_away_qb_new


## Additional Features (QB Changes, Injuries, Coach Win %)

Adds the remaining features:
- `qb_changed_home/away` — whether the starting QB changed from last week
- Injury-weighted All-Pro impact (Out players only)
- Rolling win %, points scored/allowed, and scoring differentials
- Coach cumulative win % entering the game


In [119]:
upcoming['is_playoff'] = upcoming['game_type'] != 'REG'

final_week_num = history[history['game_type'] == 'REG']['week'].max()
upcoming['is_final_week'] = (
    (upcoming['game_type'] == 'REG') &
    (upcoming['week'] == final_week_num)
)

In [120]:
# Build a lookup of who each team's QB was last week
home_qbs = history[['season', 'week', 'home_team', 'home_qb_name']].rename(
    columns={'home_team': 'team', 'home_qb_name': 'qb_name'}
)
away_qbs = history[['season', 'week', 'away_team', 'away_qb_name']].rename(
    columns={'away_team': 'team', 'away_qb_name': 'qb_name'}
)

team_qbs = pd.concat([home_qbs, away_qbs])
team_qbs = team_qbs.sort_values(['team', 'season', 'week'])

# Most recent QB per team from history
last_qb = (
    team_qbs
    .groupby('team')
    .last()
    .reset_index()
    [['team', 'qb_name']]
    .rename(columns={'qb_name': 'last_qb'})
)

# Merge onto upcoming
upcoming = upcoming.merge(
    last_qb.rename(columns={'team': 'home_team', 'last_qb': 'home_last_qb'}),
    on='home_team', how='left'
)
upcoming = upcoming.merge(
    last_qb.rename(columns={'team': 'away_team', 'last_qb': 'away_last_qb'}),
    on='away_team', how='left'
)

# Switch = QB this week is different from QB last week
upcoming['home_qb_switch'] = (
    upcoming['home_qb_name'] != upcoming['home_last_qb']
).fillna(False)
upcoming['away_qb_switch'] = (
    upcoming['away_qb_name'] != upcoming['away_last_qb']
).fillna(False)

# is_new is the same concept — your model has both
upcoming['is_home_qb_new'] = upcoming['home_qb_switch']
upcoming['is_away_qb_new'] = upcoming['away_qb_switch']

# Sanity check
print(upcoming[['home_team', 'away_team', 'home_qb_name', 'home_last_qb',
                'home_qb_switch', 'away_qb_switch']].to_string())

   home_team away_team    home_qb_name    home_last_qb  home_qb_switch  away_qb_switch
0        DEN        LV          Bo Nix          Bo Nix           False           False
1        IND       ATL    Daniel Jones    Daniel Jones           False           False
2        CAR        NO     Bryce Young     Bryce Young           False           False
3        CHI       NYG  Caleb Williams  Caleb Williams           False           False
4        HOU       JAX     Davis Mills     C.J. Stroud            True           False
5        MIA       BUF  Tua Tagovailoa  Tua Tagovailoa           False           False
6        MIN       BAL   J.J. McCarthy   J.J. McCarthy           False           False
7        NYJ       CLE    Tyrod Taylor   Justin Fields            True           False
8         TB        NE  Baker Mayfield  Baker Mayfield           False           False
9        SEA       ARI     Sam Darnold     Sam Darnold           False           False
10        SF        LA       Mac Jones     

In [121]:
# Reuse long_df from the SOS step — it already has win/loss per team per game
# If you don't have it in memory, rebuild it quickly:
home_g = history[['season', 'week', 'home_team', 'home_score', 'away_score']].copy()
home_g.columns = ['season', 'week', 'team', 'team_score', 'opp_score']
away_g = history[['season', 'week', 'away_team', 'away_score', 'home_score']].copy()
away_g.columns = ['season', 'week', 'team', 'team_score', 'opp_score']

wins_df = pd.concat([home_g, away_g])
wins_df['team_win'] = (wins_df['team_score'] > wins_df['opp_score']).astype(int)
wins_df = wins_df.sort_values(['team', 'season', 'week'])

# Rolling 5-game win pct
wins_df['rolling_win_pct'] = (
    wins_df
    .groupby('team')['team_win']
    .transform(lambda x: x.shift(1).rolling(5, min_periods=1).mean())
)

# Get most recent value per team
latest_win_pct = (
    wins_df
    .groupby('team')
    .last()
    .reset_index()
    [['team', 'rolling_win_pct']]
)

upcoming = upcoming.merge(
    latest_win_pct.rename(columns={'team': 'home_team', 'rolling_win_pct': 'home_rolling_win_pct'}),
    on='home_team', how='left'
)
upcoming = upcoming.merge(
    latest_win_pct.rename(columns={'team': 'away_team', 'rolling_win_pct': 'away_rolling_win_pct'}),
    on='away_team', how='left'
)

print(upcoming[['home_team', 'away_team', 'home_rolling_win_pct', 'away_rolling_win_pct']].to_string())

   home_team away_team  home_rolling_win_pct  away_rolling_win_pct
0        DEN        LV                   1.0                   0.2
1        IND       ATL                   0.8                   0.4
2        CAR        NO                   0.6                   0.2
3        CHI       NYG                   0.8                   0.4
4        HOU       JAX                   0.6                   0.6
5        MIA       BUF                   0.4                   0.6
6        MIN       BAL                   0.4                   0.2
7        NYJ       CLE                   0.0                   0.4
8         TB        NE                   0.6                   1.0
9        SEA       ARI                   0.8                   0.0
10        SF        LA                   0.4                   0.6
11       WAS       DET                   0.2                   0.8
12       LAC       PIT                   0.4                   0.6
13        GB       PHI                   0.6                  

In [122]:
# Scoring diff — avg points scored vs avg points allowed (rolling 5 games)
home_g2 = history[['season', 'week', 'home_team', 'away_team',
                    'home_score', 'away_score', 'spread_line', 'result']].copy()
home_g2.columns = ['season', 'week', 'team', 'opp',
                   'team_score', 'opp_score', 'spread_line', 'result']
away_g2 = history[['season', 'week', 'away_team', 'home_team',
                    'away_score', 'home_score', 'spread_line', 'result']].copy()
away_g2.columns = ['season', 'week', 'team', 'opp',
                   'team_score', 'opp_score', 'spread_line', 'result']

scoring_df = pd.concat([home_g2, away_g2])
scoring_df = scoring_df.sort_values(['team', 'season', 'week'])

# Rolling avg points scored and allowed
scoring_df['rolling_scored']  = scoring_df.groupby('team')['team_score'].transform(
    lambda x: x.shift(1).rolling(5, min_periods=1).mean()
)
scoring_df['rolling_allowed'] = scoring_df.groupby('team')['opp_score'].transform(
    lambda x: x.shift(1).rolling(5, min_periods=1).mean()
)

# Cover rate — did team cover the spread (for home perspective, result > spread means home covered)
scoring_df['covered'] = (scoring_df['result'] > scoring_df['spread_line']).astype(int)
scoring_df['rolling_cover_rate'] = scoring_df.groupby('team')['covered'].transform(
    lambda x: x.shift(1).rolling(5, min_periods=1).mean()
)

latest_scoring = (
    scoring_df.groupby('team').last().reset_index()
    [['team', 'rolling_scored', 'rolling_allowed', 'rolling_cover_rate']]
)

# League rolling avg absolute margin by week
league_margin = (
    history.groupby('week')['result']
    .apply(lambda x: x.abs().mean())
    .reset_index()
    .rename(columns={'result': 'league_rolling_avg_abs_margin_by_week'})
)
# Use the most recent week's value
latest_league_margin = league_margin.iloc[-1]['league_rolling_avg_abs_margin_by_week']
upcoming['league_rolling_avg_abs_margin_by_week'] = latest_league_margin

# Merge scoring stats
upcoming = upcoming.merge(
    latest_scoring.rename(columns={
        'team': 'home_team',
        'rolling_scored': 'home_rolling_scored',
        'rolling_allowed': 'home_rolling_allowed',
        'rolling_cover_rate': 'home_rolling_cover_rate'
    }), on='home_team', how='left'
)
upcoming = upcoming.merge(
    latest_scoring.rename(columns={
        'team': 'away_team',
        'rolling_scored': 'away_rolling_scored',
        'rolling_allowed': 'away_rolling_allowed',
        'rolling_cover_rate': 'away_rolling_cover_rate'
    }), on='away_team', how='left'
)

# Compute diff features
upcoming['scoring_diff']         = upcoming['home_rolling_scored']  - upcoming['away_rolling_scored']
upcoming['scoring_diff_reverse'] = upcoming['away_rolling_scored']  - upcoming['home_rolling_scored']
upcoming['cover_rate_diff']      = upcoming['home_rolling_cover_rate'] - upcoming['away_rolling_cover_rate']

print(upcoming[['home_team', 'away_team', 'scoring_diff', 'cover_rate_diff']].to_string())

   home_team away_team  scoring_diff  cover_rate_diff
0        DEN        LV          13.0             -0.2
1        IND       ATL          17.8             -0.2
2        CAR        NO           2.2              0.2
3        CHI       NYG           0.4             -0.2
4        HOU       JAX           6.4              0.4
5        MIA       BUF          -3.6              0.6
6        MIN       BAL           5.8              0.2
7        NYJ       CLE           1.4             -0.4
8         TB        NE          -4.4              0.0
9        SEA       ARI           8.6              0.2
10        SF        LA          -5.4              0.2
11       WAS       DET          -8.6              0.0
12       LAC       PIT          -1.2              0.0
13        GB       PHI           2.6              0.2


In [123]:
# These all come from pbp_rp which you already pulled

# Sacks — a sack is when epa < 0 and it's a pass play with a sack recorded
sack_df = pbp_rp[pbp_rp['sack'] == 1].copy()
sacks_per_game = (
    sack_df.groupby(['game_id', 'defteam'])
    .size().reset_index(name='sacks')
    .rename(columns={'defteam': 'team'})
)
sacks_per_game = sacks_per_game.merge(
    schedule[['game_id', 'week', 'season']], on='game_id', how='left'
)
sacks_per_game = sacks_per_game.sort_values(['team', 'season', 'week'])
sacks_per_game['rolling_sacks'] = sacks_per_game.groupby('team')['sacks'].transform(
    lambda x: x.shift(1).rolling(5, min_periods=1).mean()
)
latest_sacks = sacks_per_game.groupby('team').last().reset_index()[['team', 'rolling_sacks']]

# Turnovers — interceptions + fumbles lost
pbp_rp['turnover'] = ((pbp_rp['interception'] == 1) | (pbp_rp['fumble_lost'] == 1)).astype(int)
to_df = (
    pbp_rp.groupby(['game_id', 'posteam'])['turnover']
    .sum().reset_index().rename(columns={'posteam': 'team'})
)
to_df = to_df.merge(schedule[['game_id', 'week', 'season']], on='game_id', how='left')
to_df = to_df.sort_values(['team', 'season', 'week'])
to_df['rolling_turnovers'] = to_df.groupby('team')['turnover'].transform(
    lambda x: x.shift(1).rolling(5, min_periods=1).mean()
)
latest_to = to_df.groupby('team').last().reset_index()[['team', 'rolling_turnovers']]

# Third down conversion rate
pbp_rp['third_down_att'] = (pbp_rp['down'] == 3).astype(int)
pbp_rp['third_down_conv'] = ((pbp_rp['down'] == 3) & (pbp_rp['first_down'] == 1)).astype(int)
third_df = (
    pbp_rp.groupby(['game_id', 'posteam'])
    .agg(third_att=('third_down_att', 'sum'), third_conv=('third_down_conv', 'sum'))
    .reset_index().rename(columns={'posteam': 'team'})
)
third_df['third_down_rate'] = third_df['third_conv'] / third_df['third_att'].replace(0, 1)
third_df = third_df.merge(schedule[['game_id', 'week', 'season']], on='game_id', how='left')
third_df = third_df.sort_values(['team', 'season', 'week'])
third_df['rolling_third_down'] = third_df.groupby('team')['third_down_rate'].transform(
    lambda x: x.shift(1).rolling(5, min_periods=1).mean()
)
latest_third = third_df.groupby('team').last().reset_index()[['team', 'rolling_third_down']]

# Merge all onto upcoming
for lookup, home_col, away_col in [
    (latest_sacks,  'home_rolling_sacks',      'away_rolling_sacks'),
    (latest_to,     'home_rolling_turnovers',   'away_rolling_turnovers'),
    (latest_third,  'home_rolling_third_down',  'away_rolling_third_down'),
]:
    upcoming = upcoming.merge(
        lookup.rename(columns={'team': 'home_team', lookup.columns[1]: home_col}),
        on='home_team', how='left'
    )
    upcoming = upcoming.merge(
        lookup.rename(columns={'team': 'away_team', lookup.columns[1]: away_col}),
        on='away_team', how='left'
    )

# Diff features
upcoming['sack_diff']              = upcoming['home_rolling_sacks']      - upcoming['away_rolling_sacks']
upcoming['sack_diff_reverse']      = upcoming['away_rolling_sacks']      - upcoming['home_rolling_sacks']
upcoming['turnover_diff']          = upcoming['home_rolling_turnovers']  - upcoming['away_rolling_turnovers']
upcoming['turnover_diff_reverse']  = upcoming['away_rolling_turnovers']  - upcoming['home_rolling_turnovers']
upcoming['third_down_diff']        = upcoming['home_rolling_third_down'] - upcoming['away_rolling_third_down']
upcoming['third_down_diff_reverse']= upcoming['away_rolling_third_down'] - upcoming['home_rolling_third_down']

print(upcoming[['home_team', 'away_team', 'sack_diff', 'turnover_diff', 'third_down_diff']].to_string())

   home_team away_team  sack_diff  turnover_diff  third_down_diff
0        DEN        LV        1.2            0.0         0.117843
1        IND       ATL       -0.6            0.2         0.053473
2        CAR        NO       -1.2            0.2        -0.106693
3        CHI       NYG       -0.4            0.0         0.045217
4        HOU       JAX        1.4            0.2        -0.047249
5        MIA       BUF        1.4            0.8        -0.206410
6        MIN       BAL        1.0           -0.2        -0.107309
7        NYJ       CLE       -0.2           -0.2        -0.098701
8         TB        NE       -1.6            0.2         0.052635
9        SEA       ARI       -0.2            0.0         0.126573
10        SF        LA       -1.4            0.4         0.178868
11       WAS       DET       -0.6            0.2        -0.133483
12       LAC       PIT        0.4            1.0        -0.009895
13        GB       PHI       -1.6            0.2         0.024115


In [124]:
have    = [f for f in model_features if f in upcoming.columns]
missing = [f for f in model_features if f not in upcoming.columns]

print(f"Have: {len(have)}/79")
print(f"\nStill missing ({len(missing)}):")
for f in missing:
    print(f"  - {f}")

Have: 69/79

Still missing (10):
  - home_qbr_prev_year
  - away_qbr_prev_year
  - diff_qbr_prev_year
  - home_injured_count
  - away_injured_count
  - diff_injured_count
  - diff_active_allpro_weighted
  - diff_active_allpro_prev_year
  - home_coach_win_pct_prior
  - away_coach_win_pct_prior


In [147]:
pass_plays = pbp_rp[
    (pbp_rp['play_type'] == 'pass') &
    (pbp_rp['passer_player_name'].notna())
].copy()

print(pbp_rp.shape)
print(pbp_rp['season'].unique())

(34628, 375)
[2025]


In [149]:
raw_pbp = nfl.load_pbp([2024, 2025])
pbp = raw_pbp.to_pandas()

pbp_rp = pbp[
    pbp['play_type'].isin(['run', 'pass']) &
    pbp['posteam'].notna() &
    pbp['defteam'].notna()
].copy()

print(pbp_rp.shape)
print(pbp_rp['season'].unique())

(69678, 372)
[2024 2025]


In [152]:
# Cell 1 - filter to pass plays
pass_plays = pbp_rp[
    (pbp_rp['play_type'] == 'pass') &
    (pbp_rp['passer_player_name'].notna())
].copy()

# Cell 2 - aggregate per QB per season
qb_stats = (
    pass_plays
    .groupby(['season', 'posteam', 'passer_player_name'])
    .agg(
        attempts    = ('pass_attempt',   'sum'),
        completions = ('complete_pass',  'sum'),
        yards       = ('passing_yards',  'sum'),
        tds         = ('pass_touchdown', 'sum'),
        ints        = ('interception',   'sum')
    )
    .reset_index()
)

qb_stats = qb_stats[qb_stats['attempts'] >= 100]

# Cell 3 - apply passer rating formula
def passer_rating(row):
    a = max(0, min(((row['completions'] / row['attempts']) - 0.3) * 5,   2.375))
    b = max(0, min(((row['yards']       / row['attempts']) - 3)  * 0.25, 2.375))
    c = max(0, min(  row['tds']         / row['attempts']  * 20,         2.375))
    d = max(0, min(2.375 - (row['ints'] / row['attempts']  * 25),        2.375))
    return ((a + b + c + d) / 6) * 100

qb_stats['passer_rating'] = qb_stats.apply(passer_rating, axis=1)

# Cell 4 - best QB per team per season (starter = most attempts)
best_qb = (
    qb_stats
    .sort_values('attempts', ascending=False)
    .groupby(['season', 'posteam'])
    .first()
    .reset_index()
    [['season', 'posteam', 'passer_player_name', 'passer_rating']]
)

print(f"Seasons covered: {best_qb['season'].min()} - {best_qb['season'].max()}")
print(best_qb[best_qb['season'] == 2024].sort_values('passer_rating', ascending=False))

Seasons covered: 2024 - 2025
    season posteam passer_player_name  passer_rating
2     2024     BAL          L.Jackson     114.682904
10    2024     DET             J.Goff     103.277732
6     2024     CIN           J.Burrow     101.096648
29    2024      TB         B.Mayfield     100.901347
3     2024     BUF            J.Allen      98.976109
22    2024      NO             D.Carr      97.270115
19    2024     MIA       T.Tagovailoa      96.174782
25    2024     PHI            J.Hurts      94.271869
20    2024     MIN          S.Darnold      91.922628
31    2024     WAS          J.Daniels      91.525206
28    2024      SF            B.Purdy      90.810858
17    2024     LAC          J.Herbert      89.949082
16    2024      LA         M.Stafford      89.656784
9     2024     DEN              B.Nix      89.474932
11    2024      GB             J.Love      89.094165
26    2024     PIT           R.Wilson      89.074190
0     2024     ARI           K.Murray      88.859649
15    2024      K

In [153]:
PREV_SEASON = TARGET_SEASON - 1   # 2024

pr_prev = best_qb[best_qb['season'] == PREV_SEASON][['posteam', 'passer_rating']].copy()

upcoming = upcoming.merge(
    pr_prev.rename(columns={'posteam': 'home_team', 'passer_rating': 'home_qbr_prev_year'}),
    on='home_team', how='left'
)
upcoming = upcoming.merge(
    pr_prev.rename(columns={'posteam': 'away_team', 'passer_rating': 'away_qbr_prev_year'}),
    on='away_team', how='left'
)

median_pr = pr_prev['passer_rating'].median()
upcoming['home_qbr_prev_year'] = upcoming['home_qbr_prev_year'].fillna(median_pr)
upcoming['away_qbr_prev_year'] = upcoming['away_qbr_prev_year'].fillna(median_pr)
upcoming['diff_qbr_prev_year'] = upcoming['home_qbr_prev_year'] - upcoming['away_qbr_prev_year']

print(upcoming[['home_team', 'away_team',
                'home_qbr_prev_year', 'away_qbr_prev_year',
                'diff_qbr_prev_year']].to_string())

   home_team away_team  home_qbr_prev_year  away_qbr_prev_year  diff_qbr_prev_year
0        DEN        LV           89.474932           74.568318           14.906614
1        IND       ATL           58.852286           84.091702          -25.239415
2        CAR        NO           76.388889           97.270115          -20.881226
3        CHI       NYG           77.823637           73.174978            4.648660
4        HOU       JAX           79.526081           79.474044            0.052038
5        MIA       BUF           96.174782           98.976109           -2.801327
6        MIN       BAL           91.922628          114.682904          -22.760276
7        NYJ       CLE           84.466374           73.415133           11.051241
8         TB        NE          100.901347           79.801693           21.099654
9        SEA       ARI           85.814123           88.859649           -3.045526
10        SF        LA           90.810858           89.656784            1.154073
11  

In [154]:
have    = [f for f in model_features if f in upcoming.columns]
missing = [f for f in model_features if f not in upcoming.columns]

print(f"Have: {len(have)}/79")
print(f"\nStill missing ({len(missing)}):")
for f in missing:
    print(f"  - {f}")

Have: 72/79

Still missing (7):
  - home_injured_count
  - away_injured_count
  - diff_injured_count
  - diff_active_allpro_weighted
  - diff_active_allpro_prev_year
  - home_coach_win_pct_prior
  - away_coach_win_pct_prior


In [155]:
raw_inj = nfl.load_injuries(seasons=[2025])
inj_df = raw_inj.to_pandas()

print(inj_df.shape)
print(inj_df.columns.tolist())
print(inj_df['report_status'].value_counts())

(6068, 16)
['season', 'season_type', 'game_type', 'team', 'week', 'gsis_id', 'position', 'full_name', 'first_name', 'last_name', 'report_primary_injury', 'report_secondary_injury', 'report_status', 'practice_primary_injury', 'practice_secondary_injury', 'practice_status']
report_status
Out             1396
Questionable    1281
Doubtful         106
Name: count, dtype: int64


In [156]:
TARGET_WEEK = upcoming['week'].iloc[0]   # whatever week you're predicting

# Only count players listed as Out
inj_status = inj_df[
    (inj_df['report_status'] == 'Out') &
    (inj_df['week'] == TARGET_WEEK)
].copy()

injuries_by_team = (
    inj_status
    .groupby(['season', 'week', 'team'])
    .agg(injured_player_count=('full_name', 'count'))
    .reset_index()
)

# Merge home
upcoming = upcoming.merge(
    injuries_by_team.rename(columns={
        'team': 'home_team',
        'injured_player_count': 'home_injured_count'
    }).drop(columns=['season', 'week']),
    on='home_team', how='left'
)

# Merge away
upcoming = upcoming.merge(
    injuries_by_team.rename(columns={
        'team': 'away_team',
        'injured_player_count': 'away_injured_count'
    }).drop(columns=['season', 'week']),
    on='away_team', how='left'
)

upcoming[['home_injured_count', 'away_injured_count']] = (
    upcoming[['home_injured_count', 'away_injured_count']].fillna(0).astype(int)
)
upcoming['diff_injured_count'] = upcoming['home_injured_count'] - upcoming['away_injured_count']

print(upcoming[['home_team', 'away_team',
                'home_injured_count', 'away_injured_count',
                'diff_injured_count']].to_string())

   home_team away_team  home_injured_count  away_injured_count  diff_injured_count
0        DEN        LV                   3                   1                   2
1        IND       ATL                   4                   3                   1
2        CAR        NO                   1                   2                  -1
3        CHI       NYG                   4                   7                  -3
4        HOU       JAX                   7                   4                   3
5        MIA       BUF                   2                   2                   0
6        MIN       BAL                   2                   0                   2
7        NYJ       CLE                   0                   2                  -2
8         TB        NE                   4                   0                   4
9        SEA       ARI                   5                   4                   1
10        SF        LA                   2                   0                   2
11  

In [157]:
# Out players only
inj_allpro_df = inj_df[
    (inj_df['report_status'] == 'Out') &
    (inj_df['week'] == TARGET_WEEK)
].copy()
inj_allpro_df['season'] = inj_allpro_df['season'].astype(int)
inj_allpro_df['week']   = inj_allpro_df['week'].astype(int)

# Build weighted all-pro history (same 4/2/1/0.5 weights as notebook)
allpro_history = []
for yrs_back, weight in zip([0, 1, 2, 3], [4, 2, 1, 0.5]):
    temp = allpro_df.copy()
    temp['season'] = temp['Year'] + yrs_back
    temp['weight'] = weight
    allpro_history.append(temp)

allpro_weighted_hist = pd.concat(allpro_history)

# Merge to find injured players who are/were all-pros
inj_allpro_df = inj_allpro_df.merge(
    allpro_weighted_hist[['Player', 'season', 'Team', 'weight']],
    left_on=['full_name', 'season'],
    right_on=['Player', 'season'],
    how='left'
)

# Only keep rows where player was an all-pro
inj_allpro_df = inj_allpro_df[inj_allpro_df['weight'].notnull()]

# Sum weights per team per week
injured_allpro_weighted = (
    inj_allpro_df
    .groupby(['season', 'week', 'team'])['weight']
    .sum()
    .reset_index()
    .rename(columns={'weight': 'injured_allpro_weighted_count'})
)

# Merge home
upcoming = upcoming.merge(
    injured_allpro_weighted.rename(columns={'team': 'home_team'}).drop(columns=['season', 'week']),
    on='home_team', how='left'
).rename(columns={'injured_allpro_weighted_count': 'home_injured_allpro_weighted_count'})

# Merge away
upcoming = upcoming.merge(
    injured_allpro_weighted.rename(columns={'team': 'away_team'}).drop(columns=['season', 'week']),
    on='away_team', how='left'
).rename(columns={'injured_allpro_weighted_count': 'away_injured_allpro_weighted_count'})

upcoming[['home_injured_allpro_weighted_count', 'away_injured_allpro_weighted_count']] = (
    upcoming[['home_injured_allpro_weighted_count', 'away_injured_allpro_weighted_count']].fillna(0)
)

# Active allpro weighted = total weighted allpro - injured weighted allpro
upcoming['home_active_allpro_weighted'] = (
    upcoming['home_allpro_last_3_years_weighted'] - upcoming['home_injured_allpro_weighted_count']
)
upcoming['away_active_allpro_weighted'] = (
    upcoming['away_allpro_last_3_years_weighted'] - upcoming['away_injured_allpro_weighted_count']
)
upcoming['diff_active_allpro_weighted'] = (
    upcoming['home_active_allpro_weighted'] - upcoming['away_active_allpro_weighted']
)

print(upcoming[['home_team', 'away_team',
                'diff_active_allpro_weighted']].to_string())

   home_team away_team  diff_active_allpro_weighted
0        DEN        LV                         16.0
1        IND       ATL                         -2.0
2        CAR        NO                          0.0
3        CHI       NYG                          0.5
4        HOU       JAX                          0.0
5        MIA       BUF                         -9.0
6        MIN       BAL                         -7.0
7        NYJ       CLE                          1.0
8         TB        NE                          2.5
9        SEA       ARI                         -1.0
10        SF        LA                         14.0
11       WAS       DET                        -20.0
12       LAC       PIT                         -2.0
13        GB       PHI                        -31.5


In [158]:
# Prev year allpros shifted to current season
allpro_prev_year_df = allpro_df.copy()
allpro_prev_year_df['season'] = allpro_prev_year_df['Year'] + 1

inj_prev_df = inj_df[
    (inj_df['report_status'] == 'Out') &
    (inj_df['week'] == TARGET_WEEK)
].copy()
inj_prev_df['season'] = inj_prev_df['season'].astype(int)

inj_prev_df = inj_prev_df.merge(
    allpro_prev_year_df[['Player', 'season', 'Team']],
    left_on=['full_name', 'season'],
    right_on=['Player', 'season'],
    how='left'
)
inj_prev_df['is_prev_year_allpro'] = inj_prev_df['Team'].notnull().astype(int)

injured_allpro_prev = (
    inj_prev_df[inj_prev_df['is_prev_year_allpro'] == 1]
    .groupby(['season', 'week', 'team'])
    .agg(injured_allpro_prev_year_count=('full_name', 'count'))
    .reset_index()
)

# Merge home
upcoming = upcoming.merge(
    injured_allpro_prev.rename(columns={'team': 'home_team'}).drop(columns=['season', 'week']),
    on='home_team', how='left'
).rename(columns={'injured_allpro_prev_year_count': 'home_injured_allpro_prev_year_count'})

# Merge away
upcoming = upcoming.merge(
    injured_allpro_prev.rename(columns={'team': 'away_team'}).drop(columns=['season', 'week']),
    on='away_team', how='left'
).rename(columns={'injured_allpro_prev_year_count': 'away_injured_allpro_prev_year_count'})

upcoming[['home_injured_allpro_prev_year_count', 'away_injured_allpro_prev_year_count']] = (
    upcoming[['home_injured_allpro_prev_year_count', 'away_injured_allpro_prev_year_count']].fillna(0).astype(int)
)

upcoming['home_active_allpro_prev_year'] = (
    upcoming['home_allpro_prev_year'] - upcoming['home_injured_allpro_prev_year_count']
)
upcoming['away_active_allpro_prev_year'] = (
    upcoming['away_allpro_prev_year'] - upcoming['away_injured_allpro_prev_year_count']
)
upcoming['diff_active_allpro_prev_year'] = (
    upcoming['home_active_allpro_prev_year'] - upcoming['away_active_allpro_prev_year']
)

print(upcoming[['home_team', 'away_team', 'diff_active_allpro_prev_year']].to_string())

   home_team away_team  diff_active_allpro_prev_year
0        DEN        LV                           4.0
1        IND       ATL                           0.0
2        CAR        NO                           0.0
3        CHI       NYG                           0.0
4        HOU       JAX                           1.0
5        MIA       BUF                          -3.0
6        MIN       BAL                          -2.0
7        NYJ       CLE                          -1.0
8         TB        NE                           0.0
9        SEA       ARI                          -1.0
10        SF        LA                           3.0
11       WAS       DET                          -5.0
12       LAC       PIT                           0.0
13        GB       PHI                          -6.0


In [160]:
import numpy as np

# Pull extended schedule history for coach win pct
raw_coach = nfl.load_schedules(list(range(1999, 2026)))
coach_df = raw_coach.to_pandas()

# Reshape to long format — one row per team per game
home_c = coach_df[['game_id', 'season', 'week', 'home_team', 'away_team',
                    'home_score', 'away_score', 'home_coach']].copy()
home_c.rename(columns={
    'home_team': 'team', 'away_team': 'opponent',
    'home_score': 'team_score', 'away_score': 'opponent_score',
    'home_coach': 'coach'
}, inplace=True)

away_c = coach_df[['game_id', 'season', 'week', 'away_team', 'home_team',
                    'away_score', 'home_score', 'away_coach']].copy()
away_c.rename(columns={
    'away_team': 'team', 'home_team': 'opponent',
    'away_score': 'team_score', 'home_score': 'opponent_score',
    'away_coach': 'coach'
}, inplace=True)

team_games_df = pd.concat([home_c, away_c], ignore_index=True)
team_games_df = team_games_df.dropna(subset=['team_score', 'opponent_score'])
team_games_df['win'] = (team_games_df['team_score'] > team_games_df['opponent_score']).astype(int)

# Cumulative win % per coach going INTO each game (shift so no leakage)
def compute_cumulative_stats(group):
    group = group.sort_values(['season', 'week', 'game_id']).copy()
    group['cumulative_wins']  = group['win'].cumsum().shift(fill_value=0)
    group['cumulative_games'] = group['win'].expanding().count().shift(fill_value=0)
    return group

team_games_df = team_games_df.groupby('coach', group_keys=False).apply(compute_cumulative_stats)

team_games_df['coach_win_pct_prior'] = (
    team_games_df['cumulative_wins'] /
    team_games_df['cumulative_games'].replace(0, np.nan)
).fillna(0).round(3)

coach_win_pct_df = team_games_df[['game_id', 'team', 'coach', 'coach_win_pct_prior']]

# Merge onto upcoming using game_id
upcoming = upcoming.merge(
    coach_win_pct_df.rename(columns={
        'team': 'home_team',
        'coach_win_pct_prior': 'home_coach_win_pct_prior'
    })[['game_id', 'home_team', 'home_coach_win_pct_prior']],
    on=['game_id', 'home_team'], how='left'
)

upcoming = upcoming.merge(
    coach_win_pct_df.rename(columns={
        'team': 'away_team',
        'coach_win_pct_prior': 'away_coach_win_pct_prior'
    })[['game_id', 'away_team', 'away_coach_win_pct_prior']],
    on=['game_id', 'away_team'], how='left'
)

print(upcoming[['home_team', 'away_team',
                'home_coach_win_pct_prior',
                'away_coach_win_pct_prior']].to_string())

   home_team away_team  home_coach_win_pct_prior  away_coach_win_pct_prior
0        DEN        LV                     0.616                     0.581
1        IND       ATL                     0.558                     0.381
2        CAR        NO                     0.385                     0.111
3        CHI       NYG                     0.625                     0.339
4        HOU       JAX                     0.543                     0.625
5        MIA       BUF                     0.484                     0.647
6        MIN       BAL                     0.623                     0.610
7        NYJ       CLE                     0.125                     0.453
8         TB        NE                     0.462                     0.558
9        SEA       ARI                     0.640                     0.357
10        SF        LA                     0.549                     0.614
11       WAS       DET                     0.529                     0.575
12       LAC       PIT   

/tmp/ipykernel_4853/2207717588.py:35: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  team_games_df = team_games_df.groupby('coach', group_keys=False).apply(compute_cumulative_stats)


In [161]:
have    = [f for f in model_features if f in upcoming.columns]
missing = [f for f in model_features if f not in upcoming.columns]

print(f"Have: {len(have)}/79")
print(f"\nStill missing ({len(missing)}):")
for f in missing:
    print(f"  - {f}")

Have: 79/79

Still missing (0):


## Run Model & Evaluate Predictions

Assembles the final feature matrix, runs `pipeline.predict()`, and compares predicted margin vs. Vegas spread. Filters to games with edge >= 3 pts and evaluates against actual results.

In [162]:
# These are the exact columns the model was trained on
feature_cols = cat_cols + num_cols  # from step 1 at the very beginning

# Pull only those columns from upcoming
X = upcoming[feature_cols].copy()

print(X.shape)        # should be (14, 79) — 14 games, 79 features
print(X.isnull().sum().sum())   # should be 0 — no nulls

(14, 79)
0


In [163]:
pipeline = res['pipeline']

predictions = pipeline.predict(X)

print(predictions)

[ 9.085636    6.388661    6.5279517   8.103466    1.5310072  -7.4088016
 -5.252318   -1.6554894  -0.92272186  8.256675   -6.96885    -5.7116556
  0.4734495   2.5876126 ]


In [164]:
results = upcoming[['game_id', 'home_team', 'away_team',
                     'gameday', 'spread_line']].copy()

results['predicted_margin'] = predictions.round(1)

# Model edge = how much the model disagrees with the spread
# Positive = model thinks home covers, negative = model thinks away covers
results['model_edge'] = (results['predicted_margin'] - results['spread_line']).round(1)

# Recommendation
def recommend(row):
    if row['model_edge'] > 0:
        return f"BET HOME ({row['home_team']})"
    elif row['model_edge'] < 0:
        return f"BET AWAY ({row['away_team']})"
    else:
        return "PASS"

results['recommendation'] = results.apply(recommend, axis=1)

# Sort by strongest edge first
results = results.sort_values('model_edge', key=abs, ascending=False)

print(results.to_string(index=False))

        game_id home_team away_team    gameday  spread_line  predicted_margin  model_edge recommendation
2025_10_NYG_CHI       CHI       NYG 2025-11-09          4.5               8.1         3.6 BET HOME (CHI)
  2025_10_NE_TB        TB        NE 2025-11-09          2.5              -0.9        -3.4  BET AWAY (NE)
2025_10_JAX_HOU       HOU       JAX 2025-11-09         -1.5               1.5         3.0 BET HOME (HOU)
2025_10_DET_WAS       WAS       DET 2025-11-09         -8.5              -5.7         2.8 BET HOME (WAS)
2025_10_PIT_LAC       LAC       PIT 2025-11-09          3.0               0.5        -2.5 BET AWAY (PIT)
  2025_10_LA_SF        SF        LA 2025-11-09         -5.5              -7.0        -1.5  BET AWAY (LA)
2025_10_ARI_SEA       SEA       ARI 2025-11-09          7.0               8.3         1.3 BET HOME (SEA)
 2025_10_PHI_GB        GB       PHI 2025-11-10          1.5               2.6         1.1  BET HOME (GB)
2025_10_BUF_MIA       MIA       BUF 2025-11-09         

In [166]:
EDGE_THRESHOLD = 3.0   # only show games where model disagrees with spread by 1+ pts

strong_edges = results[results['model_edge'].abs() >= EDGE_THRESHOLD]

print(f"Games above threshold: {len(strong_edges)}/{len(results)}")
print()
print(strong_edges[['home_team', 'away_team', 'spread_line',
                     'predicted_margin', 'model_edge',
                     'recommendation']].to_string(index=False))

Games above threshold: 3/14

home_team away_team  spread_line  predicted_margin  model_edge recommendation
      CHI       NYG          4.5               8.1         3.6 BET HOME (CHI)
       TB        NE          2.5              -0.9        -3.4  BET AWAY (NE)
      HOU       JAX         -1.5               1.5         3.0 BET HOME (HOU)


In [167]:
# Actual results are already in your schedule DataFrame
actual = schedule[
    (schedule['season'] == 2025) &
    (schedule['week'] == 10)
][['game_id', 'home_team', 'away_team', 'home_score',
   'away_score', 'spread_line', 'result']].copy()

# result = home score - away score (actual margin)
actual = actual.rename(columns={'result': 'actual_margin'})

print(actual.to_string(index=False))

        game_id home_team away_team  home_score  away_score  spread_line  actual_margin
 2025_10_LV_DEN       DEN        LV          10           7          9.5              3
2025_10_ATL_IND       IND       ATL          31          25          6.5              6
 2025_10_NO_CAR       CAR        NO           7          17          5.5            -10
2025_10_NYG_CHI       CHI       NYG          24          20          4.5              4
2025_10_JAX_HOU       HOU       JAX          36          29         -1.5              7
2025_10_BUF_MIA       MIA       BUF          30          13         -8.5             17
2025_10_BAL_MIN       MIN       BAL          19          27         -4.5             -8
2025_10_CLE_NYJ       NYJ       CLE          27          20         -1.5              7
  2025_10_NE_TB        TB        NE          23          28          2.5             -5
2025_10_ARI_SEA       SEA       ARI          44          22          7.0             22
  2025_10_LA_SF        SF       

In [169]:
EDGE_THRESHOLD = 3.0   # only show games where model disagrees with spread by 1+ pts

strong_edges = results[results['model_edge'].abs() >= EDGE_THRESHOLD]

print(f"Games above threshold: {len(strong_edges)}/{len(results)}")
print()
print(strong_edges[['home_team', 'away_team', 'spread_line',
                     'predicted_margin', 'model_edge',
                     'recommendation']].to_string(index=False))

Games above threshold: 3/14

home_team away_team  spread_line  predicted_margin  model_edge recommendation
      CHI       NYG          4.5               8.1         3.6 BET HOME (CHI)
       TB        NE          2.5              -0.9        -3.4  BET AWAY (NE)
      HOU       JAX         -1.5               1.5         3.0 BET HOME (HOU)


In [168]:
eval_df = results.merge(
    actual[['game_id', 'actual_margin', 'home_score', 'away_score']],
    on='game_id', how='left'
)

# Did the home team actually cover the spread?
# actual_margin > spread_line means home covered
eval_df['home_covered'] = eval_df['actual_margin'] > eval_df['spread_line']

# Did the model call it correctly?
eval_df['model_correct'] = (
    (eval_df['model_edge'] > 0) == eval_df['home_covered']
)

# Prediction error — how far off was the margin prediction
eval_df['margin_error'] = (
    eval_df['predicted_margin'] - eval_df['actual_margin']
).abs().round(1)

print(eval_df[['home_team', 'away_team', 'spread_line',
               'predicted_margin', 'actual_margin',
               'model_edge', 'home_covered',
               'model_correct', 'margin_error']].to_string(index=False))

home_team away_team  spread_line  predicted_margin  actual_margin  model_edge  home_covered  model_correct  margin_error
      CHI       NYG          4.5               8.1              4         3.6         False          False           4.1
       TB        NE          2.5              -0.9             -5        -3.4         False           True           4.1
      HOU       JAX         -1.5               1.5              7         3.0          True           True           5.5
      WAS       DET         -8.5              -5.7            -22         2.8         False          False          16.3
      LAC       PIT          3.0               0.5             15        -2.5          True          False          14.5
       SF        LA         -5.5              -7.0            -16        -1.5         False           True           9.0
      SEA       ARI          7.0               8.3             22         1.3          True           True          13.7
       GB       PHI          1.5

## Week 10 Results Note

**Week 10 2025:** Model went 8/14 on all games, 2/3 on high-edge bets (≥3 pt edge). This was the initial validation run before the production pipeline was built.